# 🏭 RiskLens — Multi-Modal Product-Risk Pipeline
### Build Custom AI — SarasAI · graded capstone · **100 points · 25 per week · S ≥ 70**

**This is the only notebook you submit — every week, the same link.** You inherit a *partially
working* pipeline: the plumbing runs, the intelligence is weak or missing. Each week you replace
one stage's baseline with your own build and prove the improvement on a frozen set.

```
photo ──▶ STAGE 1 · VLM inspector ──▶ defect note (JSON)
                                          │
reviews+specs ──▶ STAGE 2 · RAG ──▶ evidence (cited)
                                          │
                              STAGE 3 · Analyst SLM (QLoRA)
                                          │
                              STAGE 4 · compressed + shipped
                                          ▼
                          root-cause risk report + eval appendix
```

| Week | You deliver | Deliverables | Points |
|---|---|---|---|
| 1 | Stage 1 — fix the baseline inspector's 3 bugs, beat it on a frozen split | D1.1–D1.7 | 25 |
| 2 | Stage 2 — corpus + gold set + a retriever that measurably beats the baseline | D2.1–D2.7 | 25 |
| 3 | Stage 3 — teacher-generated dataset + QLoRA Analyst + three-way eval + ablation | D3.1–D3.7 | 25 |
| 4 | Stage 4 — quantization gate + assembled pipeline + batch stress test + economics | D4.1–D4.7 | 25 |
| 5 | Final delivery: run-all + metrics ledger + reflection *(deductions apply if missing)* | — | — |

Full criteria, deductions and bonuses: **`RUBRICS.md`** (same folder).

## Rules of the notebook

1. **Don't restructure.** Graders navigate by the section headers and the `Dx.y` labels. Fill
   the labeled scaffold cells; add extra cells freely *below* the scaffold they extend; never
   edit cells marked `⚙️ GIVEN`.
2. **Frozen means frozen.** Each stage's frozen artifact (split / gold set / dataset card /
   gate) is authored **above** the build work — cell order is grading evidence.
3. **Run all before submitting** (`Runtime → Run all`). Graders read saved outputs; an evaluation cell without output scores as not delivered. ("Run a evaluation cell once" = once per submitted
   session; the Week-5 run-all is expected — write your training cell as *train-or-load* so
   run-all doesn't re-train.)
4. **Memory budget (16 GB T4):** the VLM stays resident all session. Keep the Analyst ≤1.5B and
   ship-arms in 4-bit, or run-all will OOM at Stage 4.
5. **Honesty outscores numbers.** An honest 78% with clean methodology beats a leaked 95% —
   eval smells carry explicit deductions (see rubric).

---
# 0 · Project Setup  `⚙️ GIVEN

Shared infrastructure for all four stages. **Read it before Week 1 — you will reuse every
function here, and your own code is expected to match this standard** (docstrings, named
constants, comments that explain *why*, not *what*).

In [ ]:

!pip install -q "transformers>=4.50" "trl>=0.17" "peft>=0.14" bitsandbytes datasets accelerate sentence-transformers faiss-cpu scikit-learn matplotlib
!pip install num2words

### Restart session after the instaltion of these libraries!

In [ ]:
# ⚙️ GIVEN · environment ---------------------------------------------------------------
import torch, json, os, re, time, random
import numpy as np

SEED = 42                                     # one seed, used everywhere, printed everywhere
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# T4s (pre-Ampere) have no native bfloat16 — everything downstream uses this DTYPE.
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16

GPU_PRICE_PER_H = 0.40                        # $/h assumed in all cost math; update to yours

assert torch.cuda.is_available(), "The capstone needs a GPU (a free T4 is enough)."
print(f" |  GPU: {torch.cuda.get_device_name(0)}  |  dtype: {DTYPE}  |  seed: {SEED}")

### ⚙️ metrics ledger · *given*

In [ ]:
# Every stage's evaluation cell logs its frozen-set numbers here. Over five weeks this file
# becomes your evaluation appendix — the evidence trail graders (and you) read.
METRICS_FILE = "capstone_metrics.json"

#### `record_metrics()`

In [ ]:
def record_metrics(stage, **metrics):
    """Append or overwrite one stage's metrics in the ledger.

    Args:
        stage:     ledger key, e.g. "stage1_vlm". Re-running a evaluation cell overwrites
                   that stage's entry only — other stages' history is preserved.
        **metrics: flat name -> number/string pairs; keep them JSON-serializable.
    """
    log = json.load(open(METRICS_FILE)) if os.path.exists(METRICS_FILE) else {}
    log[stage] = {**metrics}
    json.dump(log, open(METRICS_FILE, "w"), indent=1)
    print(f"📊 recorded [{stage}]:", metrics)

#### `show_ledger()`

In [ ]:
def show_ledger():
    """Print the full five-week evidence trail (Week 5 ends with this)."""
    print(json.dumps(json.load(open(METRICS_FILE)), indent=1)
          if os.path.exists(METRICS_FILE) else "ledger empty — no stage recorded yet")

### ⚙️ VLM plumbing · *given*

In [ ]:
# Talking to a VLM is boilerplate, not the assignment — the intelligence you put on top
# of it is. Loaded once; stays resident all session (see Rule 4).
from transformers import AutoProcessor, AutoModelForImageTextToText

VLM_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"     # swapping is allowed — justify it in D1.3

processor = AutoProcessor.from_pretrained(VLM_ID)
vlm = AutoModelForImageTextToText.from_pretrained(VLM_ID, torch_dtype=DTYPE, device_map="auto")

#### `ask_vlm()`

In [ ]:
def ask_vlm(image, question, max_new_tokens=100):
    """Send one image + one question to the VLM; return its text answer.

    Args:
        image:          a PIL.Image (RGB).
        question:       the text prompt shown alongside the image.
        max_new_tokens: generation cap — small (10) for one-word verdicts,
                        larger (80+) for JSON or descriptions.
    Returns:
        The model's reply as a stripped string (its "Assistant:" turn only).
    """
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": question}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt",
    ).to(vlm.device, dtype=DTYPE)                 # BatchFeature.to casts only float tensors
    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0], skip_special_tokens=True).split("Assistant:")[-1].strip()

print("VLM resident:", VLM_ID)

### ⚙️ retrieval + shared metrics · *given*

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")   # 384-dim, fast, strong for its size

# BGE retrieves better when the QUERY carries this instruction prefix. Documents stay bare.
BGE_PREFIX = "Represent this sentence for searching relevant passages: "

#### `build_index()`

In [ ]:
def build_index(texts):
    """Embed `texts` (list of str) and return (faiss_index, matrix).

    Vectors are L2-normalized, so the inner-product index computes cosine similarity —
    that identity is why IndexFlatIP is correct here and why normalization is not optional.
    """
    vecs = embedder.encode(texts, normalize_embeddings=True).astype(np.float32)
    ix = faiss.IndexFlatIP(vecs.shape[1])
    ix.add(vecs)
    return ix, vecs

# --- the Analyst report contract: identical in Weeks 3/4, the assignments, the holdout ---
REQUIRED_SECTIONS = ["## ROOT CAUSE ANALYSIS", "**Product:**", "**Defect:**", "**Severity:**",
                     "**Evidence:**", "**Probable cause:**", "**Recommended action:**"]

#### `format_ok()`

In [ ]:
def format_ok(text):
    """True iff ALL seven headers are present AND Severity is exactly LOW/MEDIUM/HIGH."""
    return all(h in text for h in REQUIRED_SECTIONS) and bool(
        re.search(r"\*\*Severity:\*\*\s*(LOW|MEDIUM|HIGH)", text))

#### `bench()`

In [ ]:
def bench(fn, arg, n_runs=6):
    """p50/p95 latency (seconds) of fn(arg) over n_runs.

    p95 is what users feel — SLOs are written against it, not the average.
    """
    ts = []
    for _ in range(n_runs):
        t0 = time.perf_counter(); fn(arg); ts.append(time.perf_counter() - t0)
    t = np.array(ts)
    return {"p50_s": round(float(np.percentile(t, 50)), 2),
            "p95_s": round(float(np.percentile(t, 95)), 2)}

print("retrieval + metrics library ready")

### ⚙️ the cross-stage contract · *given*

In [ ]:
# Stage 3 TRAINS on strings of exactly this shape, Stage 4 serves them.
# Do not alter either function — everything downstream depends on the format.

#### `make_input()`

In [ ]:
def make_input(defect_note, evidence):
    """Assemble the Analyst's input: prose defect note + bulleted evidence block."""
    return (f"Defect note from visual inspection:\n{defect_note}\n\n"
            f"Retrieved evidence:\n{evidence}\n\nWrite a root cause analysis report.")

#### `split_input()`

In [ ]:
def split_input(row):
    """Inverse of make_input for a stored {"input": ...} row -> (note, evidence)."""
    note = row["input"].split("Retrieved evidence:")[0].replace(
        "Defect note from visual inspection:", "").strip()
    ev = row["input"].split("Retrieved evidence:")[-1].split(
        "Write a root cause analysis report.")[0].strip()
    return note, ev

# A refusal only counts if it contains one of these — write your RULES line accordingly.
REFUSAL_PHRASES = ["don't have enough information", "do not have enough information",
                   "not enough information", "cannot answer"]

print("contract locked: make_input / split_input / REQUIRED_SECTIONS / REFUSAL_PHRASES")

In [ ]:
# ⚙️ GIVEN · CAPSTONE_DOCS — the official internal documents of the RiskLens world ------
# Your Stage-2 corpus MUST include these four, verbatim, with these exact source ids:
# they are the product world your evidence engine must master. Add your own on top (D2.1).
CAPSTONE_DOCS = [
    {"source": "spec_kettle", "kind": "doc", "text":
        "KT-2000 electric kettle. Capacity 1.7 L. Body: borosilicate glass. "
        "Lid assembly: polypropylene, part PP-114. Warranty: 24 months."},
    {"source": "spec_toaster", "kind": "doc", "text":
        "TS-400 two-slot toaster. Lever mechanism part SP-77C revised in 2023-06 "
        "to fix reported sticking. Warranty: 12 months."},
    {"source": "spec_blender", "kind": "doc", "text":
        "BL-900 blender. Motor 900 W. Jar 1.5 L with silicone lid seal, part LS-22. "
        "Jar is dishwasher safe up to 60 degrees. Warranty: 12 months on motor."},
    {"source": "policy_returns", "kind": "doc", "text":
        "Return policy for all products (KT-2000, TS-400, BL-900): defective units "
        "returnable within 30 days for full refund; after 30 days warranty claims are "
        "repair or replacement. Water damage and misuse are excluded."},
]
print(len(CAPSTONE_DOCS), "official docs — include them in your corpus unchanged")

---
# STAGE 1 · VLM Defect Inspector  `(Week 1 · 25 pts)`

**Mission:** you inherit `inspect_v0` — it *runs*, and it's *bad*, with **three planted bugs**.
Diagnose them, build a proper `inspect()`, and prove yours is better on a split you froze first.

| # | Deliverable | Where |
|---|---|---|
| D1.1 | Frozen split: ≥40 test images (≥20/class) + dev set, seed printed, frozen before any prompt work | SPLIT cell |
| D1.2 | All 3 baseline bugs diagnosed: symptom → root cause → fix, one markdown cell each | scaffolds |
| D1.3 | `inspect()` honoring the JSON contract, never raises; VLM choice justified | scaffold |
| D1.4 | Prompt ladder: ≥3 measured iterations on dev, incl. one few-shot attempt | scaffold |
| D1.5 | Error analysis: ≥3 misclassified test images shown, with hypotheses | scaffold |
| D1.6 | Evaluation cell run, outputs saved — beats `inspect_v0` on defect **F1**; cost/1k recorded | 📊 cell |
| D1.7 | Resolution-budget experiment: dev accuracy + sec/image at ≥3 image sizes, with a recommendation | scaffold |

**Suggested week plan:** Day 1–2: D1.1 + D1.2 · Day 3–4: D1.3 + D1.4 · Day 5: D1.5 + D1.7 ·
Day 6: evaluation cell + checklist. *(The live session covers everything needed for D1.1–D1.4.)*

### ⚙️ data — downloads ONLY the requested category (~150 MB, not the 5 GB repo) · *given*

In [ ]:
from huggingface_hub import snapshot_download
from PIL import Image as PILImage
import glob

#### `load_mvtec()`

In [ ]:
def load_mvtec(category="bottle", n_good=30, n_defective=30):
    """Load (image, label) pairs for one MVTec category.

    MVTec convention worth knowing: the *train* split contains only good units (the dataset
    was built for anomaly detection), so defects come from the *test* split. Files are taken
    in sorted-path order, which makes every run reproducible.

    Returns:
        list of (PIL.Image, "good"|"defective") — good first, then defective.
    """
    root = snapshot_download(
        "TheoM55/mvtec_anomaly_detection", repo_type="dataset",
        allow_patterns=[f"images/train/{category}/*", f"images/test/{category}/*"])
    good = sorted(glob.glob(f"{root}/images/train/{category}/good/*.png"))[:n_good]
    bad = sorted(p for p in glob.glob(f"{root}/images/test/{category}/*/*.png")
                 if "/good/" not in p)[:n_defective]
    return ([(PILImage.open(p).convert("RGB"), "good") for p in good] +
            [(PILImage.open(p).convert("RGB"), "defective") for p in bad])

print("adapter ready — a 2nd category or your own product photos earns a bonus (see rubric)")

### ⚙️ the baseline you must beat. It runs. It is bad ON PURPOSE. · *given*

In [ ]:
# Three planted bugs — your D1.2 diagnoses all of them:
#   BUG-1 (parser):   look at the verdict line. What does it do to "No defects found."?
#   BUG-2 (prompt):   no role, no defect checklist, no rule about reflections/glare.
#   BUG-3 (contract): defect_type and description are never filled -> Stage 2 has nothing
#                     to retrieve with, Stage 3 has nothing to analyze. Downstream starves.

#### `inspect_v0()`

In [ ]:
def inspect_v0(image):
    """Baseline inspector — weak by design; see the planted bugs above."""
    answer = ask_vlm(image, "Is this defective?", max_new_tokens=30)
    return {"verdict": "DEFECTIVE" if "DEFECT" in answer.upper() else "GOOD",
            "defect_type": None, "description": ""}

demo = inspect_v0(load_mvtec(n_good=1, n_defective=0)[0][0])
print("baseline output:", demo, "  <- note the empty fields (BUG-3 in action)")

### ✏️ D1.1 — SPLIT — author FIRST, freeze, never touch again

> Requirements: >=40 test images (>=20 per class), a separate dev set for iteration,
> the seed and class counts PRINTED (graders check this cell's output and its position).

In [ ]:
# D1.1 — your work goes here
test_images = []   # [(PIL.Image, "good"|"defective"), ...]  — frozen 🔒
dev_images = []    # iterate prompts here, and only here

raise NotImplementedError("D1.1 — build the frozen split, print seed + counts, delete this line")

### D1.2 · Bug diagnoses — one cell per bug
Replace each placeholder below. A diagnosis has three parts: **symptom** (what you observed —
show a concrete input/output), **root cause** (why the code does that), **fix** (what your
`inspect()` does differently). One short paragraph each; evidence beats length.

**BUG-1 · (your title)** — *symptom:* … · *root cause:* … · *fix:* …

**BUG-2 · (your title)** — *symptom:* … · *root cause:* … · *fix:* …

**BUG-3 · (your title)** — *symptom:* … · *root cause:* … · *fix:* …

### ✏️ D1.3 — your inspector

> Requirements checklist for this function:
> [ ] domain prompt: inspector role + defect checklist + reflections-are-not-defects rule
> [ ] structured JSON output requested from the model
> [ ] defensive parsing: regex the JSON block, normalize keys, negation-safe fallback
> [ ] never raises — wrap parsing in try/except with the safe fallback dict
> If you swapped VLM_ID: which model, why, and what it cost you (memory/speed)? One sentence.

In [ ]:
# D1.3 — your work goes here
def inspect(image):
    """Your VLM inspector — same signature as inspect_v0, all three bugs fixed.

    Returns (ALWAYS — never raises; fall back safely on any parse failure):
        {"verdict": "GOOD"|"DEFECTIVE", "defect_type": str|None, "description": str}
    """
    raise NotImplementedError("D1.3 — implement, then delete this line")

### ✏️ D1.4 — prompt ladder — >=3 measured iterations on DEV

> The ladder (add one cell per rung below — keep every measurement visible):
> rung 1: your v1 prompt               -> dev accuracy
> rung 2: your improved v2             -> dev accuracy (what changed? why?)
> rung 3: few-shot with example images -> dev accuracy + a per-call token-cost note
> Conclude with ONE line: which rung inspect() uses, and why.

In [ ]:
# D1.4 — your work goes here
def dev_accuracy(predict_fn):
    """Fraction of dev_images classified correctly by predict_fn(image) -> "good"|"defective"."""
    return sum(predict_fn(img) == truth for img, truth in dev_images) / len(dev_images)

raise NotImplementedError("D1.4 — run the ladder, then delete this line")

### ✏️ D1.5 — error analysis

> AFTER the evaluation cell has run: show >=3 misclassified TEST images (plt.imshow) and write
> one hypothesis for each — WHY did the model get it wrong? (lighting? tiny defect? prompt
> gap? parser edge case?) Analyzed misses are worth more than a perfect score.

In [ ]:
# D1.5 — your work goes here
raise NotImplementedError("D1.5 — analyze after grading, then delete this line")

### ✏️ D1.7 — resolution-budget experiment

> Image tokens dominate VLM cost. Measure the trade-off on DEV at >=3 sizes:
> for size in [256, 512, 1024]:
> resized_dev = [(img.resize((size, size)), y) for img, y in dev_images]
> -> dev accuracy + mean sec/image at each size
> Deliverable: a small printed table + ONE sentence: the size you'd run in production, why.

In [ ]:
# D1.7 — your work goes here
raise NotImplementedError("D1.7 — run the experiment, then delete this line")

### 📊 Evaluation · Stage 1 — *run when the stage is done; the saved output is what graders read*

In [ ]:
# 📊 EVALUATION · Stage 1 — run when the stage is done; the saved outputs are what graders read.
# Checks (mapped to the rubric):
#   D1.1  >=40 test images, >=20 per class
#   D1.6  your inspect() beats inspect_v0 on defect F1  (F1, not recall: BUG-1 makes the
#         baseline over-call DEFECTIVE, inflating recall by construction — that's D1.2 lore)
#   logs: accuracy, defect P/R/F1, baseline F1, sec/image, $/1k  -> ledger["stage1_vlm"]
from sklearn.metrics import classification_report, confusion_matrix

def _score_inspector(fn):
    y_true = [lbl for _, lbl in test_images]
    y_pred = ["defective" if fn(img)["verdict"] == "DEFECTIVE" else "good"
              for img, _ in test_images]
    rep = classification_report(y_true, y_pred, labels=["good", "defective"],
                                output_dict=True, zero_division=0)
    return rep, y_true, y_pred

# (manual grading) assert len(test_images) >= 40, "D1.1: need >=40 frozen test images"
_c = {l: sum(1 for _, x in test_images if x == l) for l in ("good", "defective")}
# (manual grading) assert min(_c.values()) >= 20, f"D1.1: need >=20 per class, got {_c}"

_rep_v0, *_ = _score_inspector(inspect_v0)
_t0 = time.perf_counter()
_rep, _yt, _yp = _score_inspector(inspect)
_sec = (time.perf_counter() - _t0) / len(test_images)

print(f"defect F1 — baseline: {_rep_v0['defective']['f1-score']:.0%}   "
      f"yours: {_rep['defective']['f1-score']:.0%}")
print(confusion_matrix(_yt, _yp, labels=["good", "defective"]))
# (manual grading) assert _rep["defective"]["f1-score"] > _rep_v0["defective"]["f1-score"], \
# (manual grading) "D1.6: your inspector must beat the baseline on defect F1"

record_metrics("stage1_vlm", n_test=len(test_images),
    accuracy=round(_rep["accuracy"], 3),
    defect_precision=round(_rep["defective"]["precision"], 3),
    defect_recall=round(_rep["defective"]["recall"], 3),
    defect_f1=round(_rep["defective"]["f1-score"], 3),
    baseline_defect_f1=round(_rep_v0["defective"]["f1-score"], 3),
    sec_per_image=round(_sec, 2),
    usd_per_1k=round(GPU_PRICE_PER_H / 3600 * _sec * 1000, 2))

**Week 1 submission checklist** — before sending your link:
- [ ] D1.1 split cell: seed + class counts in the saved output, positioned above all prompt work
- [ ] D1.2 three bug diagnoses with concrete evidence
- [ ] D1.4 ladder: every rung's dev number visible, few-shot attempted, decision stated
- [ ] D1.5 ≥3 analyzed misses · D1.7 resolution table + recommendation
- [ ] 📊 evaluation cell output saved (F1 beats baseline, cost logged)

---
# STAGE 2 · RAG Evidence Engine  `(Week 2 · 25 pts)`

**Mission:** the inspector now names defects — connect them to evidence. You inherit
`retrieve_v0`, a naive retriever. Build a corpus worth searching, a gold set worth trusting,
and a retriever that beats the baseline **measurably**.

| # | Deliverable | Where |
|---|---|---|
| D2.1 | Corpus: ≥100 real reviews (provenance stated) + the 4 `CAPSTONE_DOCS` verbatim + ≥2 docs you author (planted facts, one cross-doc link) | CORPUS cell |
| D2.2 | Frozen gold set: ≥20 questions (≥3 multi-hop), authored **before** any indexing | GOLD cell |
| D2.3 | `retrieve()` beats `retrieve_v0` on recall@4 over the gold set | scaffold |
| D2.4 | `rag_answer()`: grounded, cites `[source_id]`s, refuses via a `REFUSAL_PHRASES` phrase; ≥4/5 adversarial questions refused | scaffold |
| D2.5 | Faithfulness + answer-relevance scored (LLM judge or `ragas`), judge caveats stated | scaffold |
| D2.6 | Evaluation cell run, outputs saved — recall table baseline-vs-yours + refusal rate | 📊 cell |
| D2.7 | One measured retrieval upgrade beyond the prefix (chunking / reranker / hybrid / k-sweep): before/after recall table + verdict | scaffold |

**Suggested week plan:** Day 1–2: D2.1 + D2.2 · Day 3: D2.3 · Day 4: D2.4 + D2.5 ·
Day 5: D2.7 · Day 6: evaluation cell + checklist.

### ⚙️ real-review loader — streams the raw JSONL head, no legacy dataset script · *given*

#### `load_reviews()`

In [ ]:
def load_reviews(n=120, category_file="Appliances.jsonl"):
    """Stream `n` usable Amazon reviews (200-800 chars) from the raw Hub file.

    Only the head of the ~2 GB file is read — streaming stops once n rows are collected.
    Returns rows already shaped like the corpus contract: {"source", "text", "kind"}.
    """
    from datasets import load_dataset
    stream = load_dataset(
        "json",
        data_files=f"hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/{category_file}",
        split="train", streaming=True)
    rows = []
    for ex in stream:
        text = (ex.get("text") or "").strip()
        if 200 < len(text) < 800:                     # skip one-word stubs and essays
            rows.append({"source": f"review_{len(rows):03d}", "text": text, "kind": "review"})
        if len(rows) >= n:
            break
    return rows

print("review loader ready — or bring your own real source (state provenance in D2.1)")

### ✏️ D2.1 — CORPUS  +  D2.2 · GOLD — gold is authored BEFORE any index exists

> corpus rows: {"source": str, "text": str, "kind": "review"|"doc"}
> requirements: >=100 reviews + the 4 CAPSTONE_DOCS verbatim + >=2 docs you author.
> Your authored docs need planted, testable facts and ONE fact that links two documents
> (that link becomes a multi-hop gold question).
> gold rows: {"q": str, "sources": [ids that MUST be retrieved], "ref": distinctive substring,
> "multihop": True on >=3 rows}
> requirements: >=20 questions, authored by READING the corpus — not by testing retrieval.

In [ ]:
# D2.1 — your work goes here
corpus = []

gold = []

raise NotImplementedError("D2.1 + D2.2 — build corpus & gold set, then delete this line")

### ⚙️ the baseline retriever you must beat (builds on YOUR corpus, lazily) · *given*

In [ ]:
_ix_v0 = None

#### `retrieve_v0()`

In [ ]:
def retrieve_v0(query, k=4):
    """Naive baseline: no query prefix, whole documents, plain cosine top-k."""
    global _ix_v0
    if _ix_v0 is None:
        _ix_v0, _ = build_index([c["text"] for c in corpus])
    q = embedder.encode([query], normalize_embeddings=True).astype(np.float32)  # <- no prefix
    scores, idxs = _ix_v0.search(q, k)
    return [{**corpus[i], "score": float(s)} for s, i in zip(scores[0], idxs[0])]

### ✏️ D2.3 — your retriever



In [ ]:
# D2.3 — your work goes here
def retrieve(query, k=4):
    """Your retriever. Returns top-k rows: [{"source", "text", "kind", "score"}, ...].

    Minimum bar: the BGE query prefix. Where the real recall@4 gains come from is your
    D2.7 experiment — chunking, a reranker, hybrid keyword+dense, or a k/score cutoff.
    """
    raise NotImplementedError("D2.3 — implement, then delete this line")

### ✏️ D2.4 — grounded answering

> >=5 questions your corpus genuinely CANNOT answer (price? CEO? colors? release date?):

In [ ]:
# D2.4 — your work goes here
def rag_answer(question, k=4):
    """Retrieve, then answer FROM the retrieved text only.

    Returns:
        (answer_text, hits) — hits is the retrieve() output used, so citations are auditable.
    Prompt requirements: context block first, RULES second (answer only from context; cite
    [source_id] after each claim; refuse with a REFUSAL_PHRASES phrase when the context
    doesn't cover it), question last.
    """
    raise NotImplementedError("D2.4 — implement, then delete this line")

ADVERSARIAL = []

### ✏️ D2.5 — faithfulness + answer-relevance

> Score BOTH axes over the gold set — they are independent failure modes:
> faithfulness: is every claim supported by the retrieved context?
> relevance:    does the answer address the question asked?
> Use an LLM judge (state which model) or the `ragas` library. REQUIRED: one markdown
> sentence on judge bias (what does sharing weights with the generator do to the score?).

In [ ]:
# D2.5 — your work goes here
raise NotImplementedError("D2.5 — score both axes, then delete this line")

### ✏️ D2.7 — one measured retrieval upgrade

> Pick ONE lever beyond the prefix: chunking strategy, a cross-encoder reranker, hybrid
> BM25+dense, or a k/score-threshold sweep. Show the SAME recall table before and after,
> and end with one sentence: adopted or rejected, and why.

In [ ]:
# D2.7 — your work goes here
raise NotImplementedError("D2.7 — run the experiment, then delete this line")

### 📊 Evaluation · Stage 2 — *run when the stage is done; the saved output is what graders read*

In [ ]:
# 📊 EVALUATION · Stage 2 — run when the stage is done; the saved outputs are what graders read.
# Checks: D2.1 corpus composition (incl. CAPSTONE_DOCS present) · D2.2 gold size + multi-hop
#         D2.3 beats baseline on recall@4 · D2.4 refusal rate over your adversarial set
# Logs:   recall@{1,2,4,8} yours + baseline recall@4, answer-ref rate, refusal rate
# (manual grading) assert len([c for c in corpus if c["kind"] == "review"]) >= 100, "D2.1: >=100 reviews"
# (manual grading) assert len([c for c in corpus if c["kind"] != "review"]) >= 6, "D2.1: 4 official + >=2 authored docs"
_ids = {c["source"] for c in corpus}
# (manual grading) assert all(d["source"] in _ids for d in CAPSTONE_DOCS), "D2.1: CAPSTONE_DOCS must be in the corpus"
# (manual grading) assert len(gold) >= 20, "D2.2: >=20 gold questions"
# (manual grading) assert sum(1 for g in gold if g.get("multihop")) >= 3, "D2.2: >=3 multi-hop rows"
# (manual grading) assert len(ADVERSARIAL) >= 5, "D2.4: >=5 adversarial questions"

def _recall_table(fn):
    out = {}
    for k in [1, 2, 4, 8]:
        hits = sum(int(set(g["sources"]) <= {h["source"] for h in fn(g["q"], k)}) for g in gold)
        out[f"recall@{k}"] = round(hits / len(gold), 3)
    return out

_r_v0, _r = _recall_table(retrieve_v0), _recall_table(retrieve)
print("baseline:", _r_v0, "\nyours:   ", _r)
# (manual grading) assert _r["recall@4"] > _r_v0["recall@4"], "D2.3: must beat the baseline on recall@4"

_ref_ok = sum(int(g["ref"].lower() in rag_answer(g["q"])[0].lower()) for g in gold)
_refusals = sum(int(any(p in rag_answer(q)[0].lower() for p in REFUSAL_PHRASES))
                for q in ADVERSARIAL)
print(f"answer-contains-ref: {_ref_ok}/{len(gold)}   refusals: {_refusals}/{len(ADVERSARIAL)}")

record_metrics("stage2_rag", n_corpus=len(corpus), n_gold=len(gold), **_r,
               baseline_recall_at_4=_r_v0["recall@4"],
               answer_ref_rate=round(_ref_ok / len(gold), 3),
               refusal_rate=round(_refusals / len(ADVERSARIAL), 3))

**Week 2 submission checklist:**
- [ ] D2.1 provenance stated; authored docs have planted facts + one cross-doc link
- [ ] D2.2 gold set authored above the index cells (order is evidence)
- [ ] D2.5 both judge axes scored, judge-bias sentence present
- [ ] D2.7 before/after table + adopt/reject verdict
- [ ] 📊 evaluation cell output saved

---
# STAGE 3 · Fine-Tuned Analyst  `(Week 3 · 25 pts)`

**Mission:** no baseline this week — the base model IS the baseline. Build a real teacher-model
dataset, prove it's clean, train QLoRA, prove the improvement three ways, and ablate one knob.

| # | Deliverable | Where |
|---|---|---|
| D3.1 | ≥150 (input → report) pairs from a **real teacher model** (the template generator from the ungraded assignment is not accepted); inputs use `make_input()` | scaffold |
| D3.2 | Dataset card complete; dedup + decontamination shown as *actions* with counts | CARD cell + scaffold |
| D3.3 | Frozen `analyst_test_rows` ≥25 + leakage certificate (max train↔test cosine ≤ 0.90) | CARD cell |
| D3.4 | QLoRA fine-tune (train-or-load pattern); adapter saved to `analyst-lora-adapter/` | scaffold |
| D3.5 | Three-way eval: format adherence · rubric score (stronger judge, caveats) · blind **randomized** win-rate vs base | scaffold |
| D3.6 | Evaluation cell run, outputs saved; `WEEK3_RESULTS` filled from your eval | 📊 cell |
| D3.7 | Ablation: change ONE training knob (r, epochs, LR, or data size), retrain, compare on the frozen set; one-paragraph conclusion | scaffold |

**Suggested week plan:** Day 1–2: D3.1 (teacher generation is the slow part — start early) ·
Day 3: D3.2 + D3.3 · Day 4: D3.4 + D3.5 · Day 5: D3.7 · Day 6: evaluation cell + checklist.

### ✏️ D3.2 — D3.3 · DATASET CARD + frozen test rows — complete BEFORE training



In [ ]:
# D3.2 — your work goes here
DATASET_CARD = {
    "teacher_model": None,             # exact model/API that wrote the reports
    "generation_strategy": None,       # how you varied products/defects/severities/phrasing
    "n_raw": None,                     # pairs before cleaning (>=150)
    "dedup_threshold": None,           # cosine threshold used (0.90 recommended)
    "n_after_dedup": None,
    "n_dropped_decontamination": None, # train rows DROPPED for sitting too close to test rows
    "max_train_test_similarity": None, # the leakage certificate (must be <= 0.90)
    "n_train": None,
    "n_test": None,                    # >= 25
}

analyst_test_rows = []                 # >=25 frozen {"input": make_input(...), "output": report}

WEEK3_RESULTS = {                      # paste from your D3.5 eval — graded for presence & plausibility
    "rubric_score_tuned": None,        # mean 1-5 from the judge
    "rubric_score_base": None,
    "win_rate": None,                  # e.g. "17/25" — report the count, not just a percent
    "judge_model": None,               # must be stronger than / different from the generator
}

raise NotImplementedError("D3.2 + D3.3 — build the dataset + card, then delete this line")

### ✏️ D3.1 — teacher generation  +  D3.2 · hygiene actions

> Workspace for: calling your teacher model, assembling raw pairs, then the hygiene chain
> from the Week-3 assignment — dedup (embedding cosine), split, decontaminate (DROP leaky
> train rows). Print counts at every step; those numbers fill DATASET_CARD.

In [ ]:
# D3.1 — your work goes here
raise NotImplementedError("D3.1 + D3.2 — generate and clean, then delete this line")

### ✏️ D3.4 — train — train-or-load pattern (run-all must not re-train)

> Pattern for your training cell:
> if os.path.exists("analyst-lora-adapter"):    # Week-5 run-all takes this branch
> ...load 4-bit base + PeftModel.from_pretrained(...)
> else:                                         # first run trains and saves
> ...QLoRA setup (LoraConfig -> SFTTrainer)...
> model.save_pretrained("analyst-lora-adapter")
> After training: model.eval()   # re-enables the KV cache + freezes dropout for eval

In [ ]:
# D3.4 — your work goes here
def analyst_report(defect_note, evidence):
    """Your fine-tuned Analyst: (note, evidence) -> a report following REQUIRED_SECTIONS."""
    raise NotImplementedError("D3.4 — implement after training, then delete this line")

raise NotImplementedError("D3.4 — train or load, then delete this line")

### ✏️ D3.5 — three-way eval on the frozen rows

> 1) format adherence: format_ok() over your generations (deterministic, no judge)
> 2) rubric score:     a judge grades 1-5 vs the reference outputs — judge must differ from
> the generator; state its name + the bias caveat
> 3) win-rate:         blind pairwise tuned-vs-base with RANDOMIZED A/B position
> (base outputs via `with model.disable_adapter():` — chat(base,...) lies)
> Fill WEEK3_RESULTS above from these numbers.

In [ ]:
# D3.5 — your work goes here
raise NotImplementedError("D3.5 — run all three, then delete this line")

### ✏️ D3.7 — ablation — one knob, one honest comparison

> Change exactly ONE of: LoRA r · epochs · learning rate · training-set size. Retrain to a
> second adapter dir, re-run the SAME frozen-set eval, and write one paragraph: what moved,
> what didn't, and what you'd pick for production. Keep both adapters' numbers visible.

In [ ]:
# D3.7 — your work goes here
raise NotImplementedError("D3.7 — ablate, then delete this line")

### 📊 Evaluation · Stage 3 — *run when the stage is done; the saved output is what graders read*

In [ ]:
# 📊 EVALUATION · Stage 3 — run when the stage is done; the saved outputs are what graders read.
# Checks: D3.1 n_raw >= 150 · D3.2 card complete · D3.3 leakage <= 0.90, >=25 frozen rows
#         D3.5 WEEK3_RESULTS filled · format adherence computed here on the frozen rows
# (manual grading) assert all(v is not None for v in DATASET_CARD.values()), "D3.2: dataset card incomplete"
# (manual grading) assert DATASET_CARD["n_raw"] >= 150, "D3.1: >=150 teacher-generated pairs"
# (manual grading) assert DATASET_CARD["max_train_test_similarity"] <= 0.90, "D3.3: leakage certificate failed"
# (manual grading) assert len(analyst_test_rows) >= 25, "D3.3: >=25 frozen test rows"
# (manual grading) assert all(v is not None for v in WEEK3_RESULTS.values()), "D3.5: fill WEEK3_RESULTS"

_outs = [analyst_report(*split_input(r)) for r in analyst_test_rows]
_fmt = sum(map(format_ok, _outs)) / len(_outs)
print(f"format adherence on frozen set: {_fmt:.0%}")

record_metrics("stage3_analyst", n_test=len(analyst_test_rows),
               format_adherence=round(_fmt, 3),
               n_train=DATASET_CARD["n_train"],
               max_train_test_sim=DATASET_CARD["max_train_test_similarity"],
               **{k: v for k, v in WEEK3_RESULTS.items() if isinstance(v, (int, float))})

**Week 3 submission checklist:**
- [ ] D3.1 teacher named; generation strategy described; ≥150 raw pairs
- [ ] D3.2 hygiene shown as actions (counts printed at each step)
- [ ] D3.5 judge named + bias caveat; win-rate randomized (say so); counts not just %
- [ ] D3.7 ablation paragraph with both adapters' numbers
- [ ] 📊 evaluation cell output saved; adapter dir (or Hub link) stated

---
# STAGE 4 · Compress, Gate & Assemble  `(Week 4 · 25 pts)`

**Mission:** make the Analyst cheap enough to ship, prove compression didn't break it, wire all
three stages into one pipeline, and find out where it strains.

| # | Deliverable | Where |
|---|---|---|
| D4.1 | GATE committed **before** any re-score: thresholds + GPU price + arm to ship | GATE cell |
| D4.2 | ≥2 compression arms with measured memory footprints | scaffold |
| D4.3 | Frozen-set re-score: format **and** content score vs Stage 3; verdict stated even if ❌ | scaffold + 🔒 |
| D4.4 | Honest speed: p50/p95 + true-token tok/s per arm; $/1k; hardware-dependence sentence | scaffold |
| D4.5 | `analyze_product()` composing Stages 1→2→3, demoed on ≥3 products (≥1 good, ≥2 defective) | scaffold |
| D4.6 | Deployment note: target runtime, what breaks at 100× traffic, the monitoring signal you'd watch | markdown |
| D4.7 | Batch stress test: full pipeline over ≥10 products; end-to-end p50/p95, stage-time breakdown, failure taxonomy | scaffold |

**Suggested week plan:** Day 1: D4.1 + D4.2 · Day 2: D4.3 + D4.4 · Day 3–4: D4.5 + D4.7 ·
Day 5: D4.6 · Day 6: evaluation cell + checklist.

### ✏️ D4.1 — GATE — commit BEFORE measuring; do not touch after the re-score



In [ ]:
# D4.1 — your work goes here
GATE = {
    "max_format_drop_pts": None,      # e.g. 5  (percentage points vs Stage 3's format rate)
    "max_content_drop_pts": None,     # e.g. 5  (points of mean cosine-to-reference, x100)
    "gpu_price_per_h": GPU_PRICE_PER_H,
    "arm_to_ship": None,              # e.g. "4-bit-nf4" — name the arm you intend to ship
}

raise NotImplementedError("D4.1 — commit the gate, then delete this line")

### ✏️ D4.2 — compression arms

> Produce >=2 arms of YOUR Analyst (e.g. 8-bit LLM.int8 and 4-bit NF4 via BitsAndBytesConfig;
> a GGUF export counts as an arm and earns the bonus). For each arm print:
> get_memory_footprint()/1e9 GB  +  the compression ratio vs half precision.

In [ ]:
# D4.2 — your work goes here
raise NotImplementedError("D4.2 — build the arms, then delete this line")

### ✏️ D4.3 — the shipped-arm contract



In [ ]:
# D4.3 — your work goes here
def analyst_report_compressed(defect_note, evidence):
    """Same contract as analyst_report, served by GATE["arm_to_ship"]."""
    raise NotImplementedError("D4.3 — implement, then delete this line")

### ✏️ D4.4 — honest speed + economics

> For the original and each arm: p50/p95 (use bench()) + tokens/sec computed from tokens
> the model ACTUALLY generated (len(tok.encode(output)) — never max_new_tokens). Then $/1k
> reports at GATE["gpu_price_per_h"]. End with one sentence: WHY can 4-bit be slower on a
> T4 while 3x smaller? (dequant overhead — the session's verified finding)

In [ ]:
# D4.4 — your work goes here
raise NotImplementedError("D4.4 — measure, then delete this line")

### ✏️ D4.5 — the assembled pipeline

> Demo on >=3 products (>=1 good, >=2 defective) in cells below — full outputs saved.

In [ ]:
# D4.5 — your work goes here
def analyze_product(image):
    """The full RiskLens pipeline: photo in, risk report out.

    Stage 1: inspect(image)                  -> defect note dict (GOOD short-circuits)
    Stage 2: retrieve(defect-based query)    -> evidence rows -> "- source: text" bullets
    Stage 3: analyst_report_compressed(...)  -> the report. The note goes in as PROSE —
             the shape it was trained on (e.g. f"{defect_type}: {description}").
    Returns:
        {"note": dict, "evidence": str, "report": str}
    """
    raise NotImplementedError("D4.5 — compose, then delete this line")

### ✏️ D4.7 — batch stress test

> Run analyze_product over >=10 test images end-to-end. Deliver:
> 1. end-to-end p50/p95 per product
> 2. stage-time breakdown (VLM vs retrieval vs Analyst — where does the time go?)
> 3. a failure taxonomy: for every product whose report is wrong/weak, WHICH stage broke?
> (this is why the stages have separate evals — prove it with your own numbers)

In [ ]:
# D4.7 — your work goes here
raise NotImplementedError("D4.7 — stress it, then delete this line")

### D4.6 · Deployment note *(replace this cell's text)*
Where would RiskLens run in production (GGUF/CPU? vLLM on GPU? a Space?), what breaks first at
100× traffic (name the stage and the resource), and the ONE monitoring signal you'd page on.

### 📊 Evaluation · Stage 4 — *run when the stage is done; the saved output is what graders read*

In [ ]:
# 📊 EVALUATION · Stage 4 — run when the stage is done; the saved outputs are what graders read.
# Checks: D4.1 gate committed · D4.3 gate verdict on the frozen rows (format + content)
#         D4.5 analyze_product honors its contract on a real image
# Logs:   post-compression metrics + gate verdict + pipeline latency -> ledger["stage4_ship"]
# (manual grading) assert all(v is not None for v in GATE.values()), "D4.1: commit the full gate first"

_outs_c = [analyst_report_compressed(*split_input(r)) for r in analyst_test_rows]
_fmt_c = sum(map(format_ok, _outs_c)) / len(_outs_c)

_refs = embedder.encode([r["output"] for r in analyst_test_rows], normalize_embeddings=True)
def _content(outs):
    g = embedder.encode(outs, normalize_embeddings=True)
    return float((g * _refs).sum(axis=1).mean())
_outs_orig = [analyst_report(*split_input(r)) for r in analyst_test_rows]
_con_o, _con_c = _content(_outs_orig), _content(_outs_c)

_prev_fmt = json.load(open(METRICS_FILE))["stage3_analyst"]["format_adherence"]
_fmt_drop = (_prev_fmt - _fmt_c) * 100
_con_drop = (_con_o - _con_c) * 100
_ok = _fmt_drop <= GATE["max_format_drop_pts"] and _con_drop <= GATE["max_content_drop_pts"]
print(f"format {_prev_fmt:.0%} -> {_fmt_c:.0%} (Δ{_fmt_drop:+.1f})   "
      f"content {_con_o:.3f} -> {_con_c:.3f} (Δ{_con_drop:+.1f})   gate: {'PASS ✅' if _ok else 'FAIL ❌'}")

_stats = bench(lambda _: analyst_report_compressed(*split_input(analyst_test_rows[0])), None, n_runs=4)
_demo = analyze_product(test_images[0][0])
# (manual grading) assert set(_demo) >= {"note", "evidence", "report"}, "D4.5: analyze_product contract"

record_metrics("stage4_ship", arm=GATE["arm_to_ship"],
               format_after=round(_fmt_c, 3), format_drop_pts=round(_fmt_drop, 1),
               content_drop_pts=round(_con_drop, 1),
               gate_verdict="PASS" if _ok else "FAIL",
               analyst_p50_s=_stats["p50_s"], analyst_p95_s=_stats["p95_s"],
               usd_per_1k=round(GATE["gpu_price_per_h"] / 3600 * _stats["p50_s"] * 1000, 2))

FINAL_REFLECTION = """
(<=300 words: architecture trade-offs · the number you trust most and least, and why ·
what you'd build next with two more weeks · what you'd tell a company adopting this pipeline)
"""

show_ledger()   # five weeks of frozen-set evidence — your evaluation appendix

#### `build_predictions()`

In [ ]:
FINAL_REFLECTION = """
(<=300 words: architecture trade-offs · the number you trust most and least, and why ·
what you'd build next with two more weeks · what you'd tell a company adopting this pipeline)
"""

show_ledger()   # five weeks of frozen-set evidence — your evaluation appendix